# Data Cleaning -- Training Dataset
**This notebook prepares Flatiron Health data for patients with advanced melanoma in anticipation of training a gradient-boosted survival model to predict mortality from the start of first-line treatment. Specifically, it processes and cleans the training cohort. Prior to data cleaning, the dataset is randomly split into training (80%) and testing (20%) subsets.**

## Import packages

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from flatiron_cleaner import DataProcessorMelanoma, merge_dataframes

## Split into train and test sets

In [2]:
df = pd.read_csv('../data/LineOfTherapy.csv')

In [3]:
df = (
    df
    .query('LineNumber == 1')
    .query('IsMaintenanceTherapy == False')
    [['PatientID', 'StartDate']]
)

In [4]:
df.shape

(9050, 2)

In [5]:
processor = DataProcessorMelanoma()

In [6]:
mortality_df = processor.process_mortality(file_path = '../data/Enhanced_Mortality_V2.csv',
                                           index_date_df = df, 
                                           index_date_column = 'StartDate',
                                           visit_path = '../data/Visit.csv', 
                                           telemedicine_path = '../data/Telemedicine.csv', 
                                           biomarkers_path = '../data/Enhanced_AdvMelanomaBiomarkers.csv', 
                                           oral_path = '../data/Enhanced_AdvMelanoma_Orals.csv',
                                           progression_path = '../data/Enhanced_AdvMelanoma_Progression.csv',
                                           metastatic_sites_path = '../data/Enhanced_AdvMelanoma_SitesOfMet.csv',
                                           drop_dates = False)

2025-11-27 00:17:12,657 - INFO - Successfully read Enhanced_Mortality_V2.csv file with shape: (7284, 2) and unique PatientIDs: 7284
2025-11-27 00:17:12,667 - INFO - Successfully merged Enhanced_Mortality_V2.csv df with index_date_df resulting in shape: (9050, 3) and unique PatientIDs: 9050
2025-11-27 00:17:13,188 - INFO - The following columns ['last_visit_date', 'last_biomarker_date', 'last_oral_date', 'last_progression_date', 'last_met_date'] are used to calculate the last EHR date
2025-11-27 00:17:13,195 - INFO - Successfully processed Enhanced_Mortality_V2.csv file with final shape: (9050, 6) and unique PatientIDs: 9050. There are 0 out of 9050 patients with missing duration values


In [7]:
df = pd.merge(df, mortality_df[['PatientID', 'event']], on = 'PatientID', how = 'left')

In [8]:
df.shape

(9050, 3)

In [9]:
# Stratify split on event 
train, test = train_test_split(
    df,
    test_size = 0.2,
    stratify = df['event'],  
    random_state = 42
)

In [10]:
train.shape

(7240, 3)

In [11]:
test.shape

(1810, 3)

In [12]:
train[['PatientID']].to_csv('../outputs/train_patient_ids.csv', index = False)
test[['PatientID']].to_csv('../outputs/test_patient_ids.csv', index = False)

## Clean CSV files 

In [13]:
train_ids = pd.read_csv('../outputs/train_patient_ids.csv')

In [14]:
train_ids = train_ids.PatientID.to_list()

In [15]:
index_date_df = df[df.PatientID.isin(train_ids)]

In [16]:
index_date_df.shape

(7240, 3)

In [17]:
index_date_df = index_date_df[['PatientID', 'StartDate']]

### Process Enhanced_AdvancedMelanoma.csv

In [18]:
enhanced_df = processor.process_enhanced(file_path = '../data/Enhanced_AdvancedMelanoma.csv',
                                         index_date_df = index_date_df,
                                         index_date_column = 'StartDate',
                                         drop_dates = False)

2025-11-27 00:17:13,290 - INFO - Successfully read Enhanced_AdvancedMelanoma.csv file with shape: (16997, 18) and unique PatientIDs: 16997
2025-11-27 00:17:13,296 - INFO - Successfully filtered Enhanced_AdvancedMelanoma.csv file with shape: (7240, 19) and unique PatientIDs: 7240
2025-11-27 00:17:13,314 - INFO - Successfully processed Enhanced_AdvancedMelanoma.csv file with final shape: (7240, 21) and unique PatientIDs: 7240


In [19]:
enhanced_df['days_adv_to_treatment'] = (enhanced_df['imported_StartDate'] - enhanced_df['AdvancedDiagnosisDate']).dt.days
enhanced_df['days_adv_to_treatment'] = np.where(enhanced_df['days_adv_to_treatment'] < 0 , 0, enhanced_df['days_adv_to_treatment'])

In [20]:
enhanced_df['days_diagnosis_to_adv'] = np.where(enhanced_df['days_diagnosis_to_adv'] < 0 , 0, enhanced_df['days_diagnosis_to_adv'])
enhanced_df['days_diagnosis_to_adv'] = enhanced_df['days_diagnosis_to_adv'].fillna(0)

In [21]:
for var in ['CalcResectInitialDx', 'CalcResectLocalRecur_mod']:
    enhanced_df[var] = enhanced_df[var].fillna('Unknown')

In [22]:
for var in ['TStage_mod', 'NStage_mod', 'MStage_mod', 'ResidualDiseaseInitialDx_mod', 'ResidualDiseaseLocalRecur_mod']:
    enhanced_df[var] = enhanced_df[var].fillna('unknown')

In [23]:
enhanced_df.GroupStage_mod.value_counts(dropna = False)

GroupStage_mod
IV         2157
III        1938
unknown    1263
II         1209
I           673
Name: count, dtype: int64

In [24]:
enhanced_df['GroupStage_mod'] = enhanced_df["GroupStage_mod"].map({
    '0': 0, 
    'I': 1,
    'II': 2,
    'III': 3,
    'IV': 4,
    'unknown': np.nan
})

enhanced_df['GroupStage_mod_na'] = np.where(enhanced_df['GroupStage_mod'].isna(), 1, 0)

# impute 4 since stage IV is most common
enhanced_df['GroupStage_mod'] = enhanced_df['GroupStage_mod'].fillna(4)

In [25]:
# drop dates
enhanced_df = enhanced_df.drop(columns = ['DiagnosisDate', 
                                          'AdvancedDiagnosisDate', 
                                          'MetDiagnosisDate', 
                                          'FirstLocalRecurDate',
                                          'FirstDistantRecurDate',
                                          'FirstVisceralMetDate',
                                          'FirstNonVisceralMetDate',
                                          'imported_StartDate',
                                          'adv_diagnosis_year', 
                                          'met_diagnosis_year',
                                          'days_diagnosis_to_met'])

### Process Demographics.csv 

In [26]:
demographics_df = processor.process_demographics(file_path = '../data/Demographics.csv',
                                                 index_date_df = index_date_df,
                                                 index_date_column = 'StartDate')

2025-11-27 00:17:13,354 - INFO - Successfully read Demographics.csv file with shape: (16997, 6) and unique PatientIDs: 16997
2025-11-27 00:17:13,362 - WARNING - Found 6 ages outside valid range (18-120)
2025-11-27 00:17:13,368 - INFO - Successfully processed Demographics.csv file with final shape: (7240, 6) and unique PatientIDs: 7240


In [27]:
demographics_df = demographics_df.query('age >= 18 or age <= 120', engine = 'python')

In [28]:
demographics_df.Gender.value_counts(dropna = False)

Gender
M      4852
F      2387
NaN       1
Name: count, dtype: int64

In [29]:
# Impute missing with male
demographics_df['sex_male'] = np.where(demographics_df['Gender'] == 'F', 0, 1)

In [30]:
demographics_df = demographics_df.drop(columns = ['Gender'])

### Process Enhanced_AdvMelanomaBiomarkers.csv

In [31]:
biomarkers_df = processor.process_biomarkers(file_path = '../data/Enhanced_AdvMelanomaBiomarkers.csv',
                                             index_date_df = index_date_df, 
                                             index_date_column = 'StartDate',
                                             days_before = None, 
                                             days_after = 30)

2025-11-27 00:17:13,433 - INFO - Successfully read Enhanced_AdvMelanomaBiomarkers.csv file with shape: (31257, 18) and unique PatientIDs: 13249
2025-11-27 00:17:13,453 - INFO - Successfully merged Enhanced_AdvMelanomaBiomarkers.csv df with index_date_df resulting in shape: (16911, 19) and unique PatientIDs: 6676
2025-11-27 00:17:13,572 - INFO - Successfully processed Enhanced_AdvMelanomaBiomarkers.csv file with final shape: (7240, 6) and unique PatientIDs: 7240


In [32]:
for var in ['BRAF_status', 'NRAS_status', 'KIT_status', 'PDL1_status']:
    biomarkers_df[var] = biomarkers_df[var].fillna('unknown')

### Process ECOG.csv

In [33]:
ecog_df = processor.process_ecog(file_path = '../data/ECOG.csv', 
                                 index_date_df = index_date_df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 days_before_further = 180)

2025-11-27 00:17:13,656 - INFO - Successfully read ECOG.csv file with shape: (211808, 4) and unique PatientIDs: 12139
2025-11-27 00:17:13,705 - INFO - Successfully merged ECOG.csv df with index_date_df resulting in shape: (123022, 5) and unique PatientIDs: 5572
2025-11-27 00:17:13,784 - INFO - Successfully processed ECOG.csv file with final shape: (7240, 3) and unique PatientIDs: 7240


In [34]:
ecog_df.ecog_index.value_counts(dropna = False)

ecog_index
NaN    2700
0      2184
1      1716
2       490
3       139
4        11
Name: count, dtype: int64

In [35]:
ecog_df['ecog_index'] = ecog_df['ecog_index'].astype('float64')
ecog_df['ecog_index_na'] = np.where(ecog_df['ecog_index'].isna(), 1, 0)

# impute 0 for missing ECOG since most common
ecog_df['ecog_index'] = ecog_df['ecog_index'].fillna(0)

In [36]:
ecog_df['ecog_newly_gte2'] = ecog_df['ecog_newly_gte2'].fillna(0)

### Process Vitals.csv

In [37]:
vitals_df = processor.process_vitals(file_path = '../data/Vitals.csv',
                                     index_date_df = index_date_df,
                                     index_date_column = 'StartDate',
                                     weight_days_before = 90,
                                     days_after = 0,
                                     vital_summary_lookback = 180, 
                                     abnormal_reading_threshold = 1)

2025-11-27 00:17:17,791 - INFO - Successfully read Vitals.csv file with shape: (4009376, 16) and unique PatientIDs: 16978
2025-11-27 00:17:19,584 - INFO - Successfully merged Vitals.csv df with index_date_df resulting in shape: (2298574, 17) and unique PatientIDs: 7238
2025-11-27 00:17:20,554 - INFO - Successfully processed Vitals.csv file with final shape: (7240, 8) and unique PatientIDs: 7240


### Process Lab.csv

In [38]:
labs_df = processor.process_labs(file_path = '../data/Lab.csv',
                                 index_date_df = index_date_df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 summary_lookback = 180)

2025-11-27 00:17:34,283 - INFO - Successfully read Lab.csv file with shape: (10819406, 17) and unique PatientIDs: 16150
2025-11-27 00:17:38,282 - INFO - Successfully merged Lab.csv df with index_date_df resulting in shape: (6435056, 18) and unique PatientIDs: 7148
2025-11-27 00:17:52,108 - INFO - Successfully processed Lab.csv file with final shape: (7240, 81) and unique PatientIDs: 7240


### Process MedicationAdministration.csv

In [39]:
medications_df = processor.process_medications(file_path = '../data/MedicationAdministration.csv',
                                               index_date_df = index_date_df,
                                               index_date_column = 'StartDate',
                                               days_before = 90,
                                               days_after = 0)

2025-11-27 00:17:53,023 - INFO - Successfully read MedicationAdministration.csv file with shape: (712643, 11) and unique PatientIDs: 12796
2025-11-27 00:17:53,276 - INFO - Successfully merged MedicationAdministration.csv df with index_date_df resulting in shape: (426461, 12) and unique PatientIDs: 6659
2025-11-27 00:17:53,322 - INFO - Successfully processed MedicationAdministration.csv file with final shape: (7240, 9) and unique PatientIDs: 7240


### Process Diagnosis.csv

In [40]:
diagnosis_df = processor.process_diagnosis(file_path = '../data/Diagnosis.csv',
                                           index_date_df = index_date_df,
                                           index_date_column = 'StartDate',
                                           days_before = None,
                                           days_after = 0)

2025-11-27 00:17:54,040 - INFO - Successfully read Diagnosis.csv file with shape: (1130923, 6) and unique PatientIDs: 16997
2025-11-27 00:17:54,254 - INFO - Successfully merged Diagnosis.csv df with index_date_df resulting in shape: (630757, 7) and unique PatientIDs: 7240
2025-11-27 00:17:55,126 - INFO - Successfully processed Diagnosis.csv file with final shape: (7240, 31) and unique PatientIDs: 7240


### Process Enhanced_Mortality_V2.csv

In [41]:
mortality_df = processor.process_mortality(file_path = '../data/Enhanced_Mortality_V2.csv',
                                           index_date_df = index_date_df, 
                                           index_date_column = 'StartDate',
                                           visit_path = '../data/Visit.csv', 
                                           telemedicine_path = '../data/Telemedicine.csv', 
                                           biomarkers_path = '../data/Enhanced_AdvMelanomaBiomarkers.csv', 
                                           oral_path = '../data/Enhanced_AdvMelanoma_Orals.csv',
                                           progression_path = '../data/Enhanced_AdvMelanoma_Progression.csv',
                                           metastatic_sites_path = '../data/Enhanced_AdvMelanoma_SitesOfMet.csv',
                                           drop_dates = False)

2025-11-27 00:17:55,146 - INFO - Successfully read Enhanced_Mortality_V2.csv file with shape: (7284, 2) and unique PatientIDs: 7284
2025-11-27 00:17:55,155 - INFO - Successfully merged Enhanced_Mortality_V2.csv df with index_date_df resulting in shape: (7240, 3) and unique PatientIDs: 7240
2025-11-27 00:17:55,643 - INFO - The following columns ['last_visit_date', 'last_biomarker_date', 'last_oral_date', 'last_progression_date', 'last_met_date'] are used to calculate the last EHR date
2025-11-27 00:17:55,649 - INFO - Successfully processed Enhanced_Mortality_V2.csv file with final shape: (7240, 6) and unique PatientIDs: 7240. There are 0 out of 7240 patients with missing duration values


In [42]:
mortality_df = mortality_df[['PatientID', 'event', 'duration']]

In [43]:
mortality_df = mortality_df.query('duration >= 0')

### Process Enhanced_AdvMelanomaProcedures.csv

In [44]:
procedures_df = processor.process_procedures(file_path = '../data/Enhanced_AdvMelanomaProcedures.csv',
                                             index_date_df = index_date_df,
                                             index_date_column = 'StartDate',
                                             days_before = None,
                                             days_after = 0)

2025-11-27 00:17:55,691 - INFO - Successfully read Enhanced_AdvMelanomaProcedures.csv file with shape: (44284, 3) and unique PatientIDs: 13432
2025-11-27 00:17:55,704 - INFO - Successfully merged Enhanced_AdvMelanomaProcedures.csv df with index_date_df resulting in shape: (15906, 4) and unique PatientIDs: 5035
2025-11-27 00:17:55,718 - INFO - Successfully processed Enhanced_AdvMelanomaProcedures.csv file with final shape: (7240, 4) and unique PatientIDs: 7240


### Process Enhanced_AdvMelanoma_SitesOfMet.csv

In [45]:
metastasis_df = processor.process_metastasis(file_path = '../data/Enhanced_AdvMelanoma_SitesOfMet.csv',
                                             index_date_df = index_date_df,
                                             index_date_column = 'StartDate',
                                             days_before = None,
                                             days_after = 0)

2025-11-27 00:17:55,732 - INFO - Successfully read Enhanced_AdvMelanoma_SitesOfMet.csv file with shape: (29302, 3) and unique PatientIDs: 10076
2025-11-27 00:17:55,742 - INFO - Successfully merged Enhanced_AdvMelanoma_SitesOfMet.csv df with index_date_df resulting in shape: (19173, 4) and unique PatientIDs: 6250
2025-11-27 00:17:55,762 - INFO - Successfully processed Enhanced_AdvMelanoma_SitesOfMet.csv file with final shape: (7240, 9) and unique PatientIDs: 7240


## Merge dataframes

In [46]:
training_df = merge_dataframes(enhanced_df,
                               demographics_df,
                               biomarkers_df,
                               ecog_df,
                               vitals_df,
                               labs_df,
                               medications_df,
                               diagnosis_df, 
                               mortality_df,
                               procedures_df,
                               metastasis_df,
                               merge_type = 'inner')

2025-11-27 00:17:55,765 - INFO - Anticipated number of merges: 10
2025-11-27 00:17:55,766 - INFO - Anticipated number of columns in final dataframe presuming all columns are unique except for PatientID: 163
2025-11-27 00:17:55,767 - INFO - Dataset 1 shape: (7240, 12), unique PatientIDs: 7240
2025-11-27 00:17:55,769 - INFO - Dataset 2 shape: (7240, 6), unique PatientIDs: 7240
2025-11-27 00:17:55,770 - INFO - Dataset 3 shape: (7240, 6), unique PatientIDs: 7240
2025-11-27 00:17:55,771 - INFO - Dataset 4 shape: (7240, 4), unique PatientIDs: 7240
2025-11-27 00:17:55,773 - INFO - Dataset 5 shape: (7240, 8), unique PatientIDs: 7240
2025-11-27 00:17:55,774 - INFO - Dataset 6 shape: (7240, 81), unique PatientIDs: 7240
2025-11-27 00:17:55,775 - INFO - Dataset 7 shape: (7240, 9), unique PatientIDs: 7240
2025-11-27 00:17:55,776 - INFO - Dataset 8 shape: (7240, 31), unique PatientIDs: 7240
2025-11-27 00:17:55,777 - INFO - Dataset 9 shape: (7225, 3), unique PatientIDs: 7225
2025-11-27 00:17:55,778 -

In [47]:
training_df.shape

(7225, 163)

## Export dataframe

In [48]:
training_df.to_csv('../outputs/1L_features_training.csv', index = False)

In [49]:
# Save dtypes
training_df.dtypes.apply(lambda x: x.name).to_csv('../outputs/1L_features_training_dtypes.csv')